<a href="https://colab.research.google.com/github/Minhaj2003/Minhaj2003/blob/main/template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install required libraries (only needed in Colab)

# requests → used to send HTTP request to a website (like opening a webpage via code)
# beautifulsoup4 → used to parse HTML and extract useful data from web pages
# openai → used to interact with AI model (for generating insights)

!pip install beautifulsoup4 requests openai

In [5]:
# Import libraries so we can use them in our code

import requests
# requests → sends request to website and gets HTML content

from bs4 import BeautifulSoup
# BeautifulSoup → helps us read and extract data from HTML (like finding text, links)

import re
# re (regular expressions) → used to find patterns like emails, phone numbers

import json
# json → used to convert data into proper JSON format (required output format)

from openai import OpenAI
# OpenAI → used to send text to AI model and get intelligent responses

In [7]:
# Function to get clean text from a website
def get_page_text(url):
    try:
        # headers → helps avoid blocking (pretend like a real browser)
        headers = {"User-Agent": "Mozilla/5.0"}

        # Send request to website
        response = requests.get(url, headers=headers, timeout=10)

        # Parse HTML content
        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unnecessary elements (scripts, styles)
        # These don't contain useful visible text
        for script in soup(["script", "style"]):
            script.extract()

        # Extract visible text only
        text = soup.get_text(separator=" ", strip=True)

        return text

    except Exception as e:
        # If something fails (bad URL, timeout, etc.)
        print(f"Error fetching {url}: {e}")
        return ""


# Function to extract emails using regex
def extract_emails(text):
    # Pattern to detect emails
    emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)

    # Remove duplicates using set
    return list(set(emails))


# Function to extract phone numbers
def extract_phone(text):
    # Pattern to detect phone numbers
    phones = re.findall(r"\+?\d[\d\s\-]{8,}\d", text)

    # Return first phone if found, else N/A
    return phones[0] if phones else "N/A"

In [11]:
# Create OpenAI client using your API key
client = OpenAI(api_key=API_KEY)

In [12]:
# ================================
# 🏆 Hackathon Template Notebook
# Prospect Research Agent
# ================================

# ========= CONFIG =========
# 🔑 Add your API key here
API_KEY = "YOUR_API_KEY"



# ========= REQUIRED FUNCTION =========
def enrich_company(url: str) -> dict:
    """
    Input: Company URL
    Output: Structured company profile (STRICT FORMAT)
    """

    # 1. Scrape website text
    text = get_page_text(url)

    # 2. Extract emails and phone numbers
    emails = extract_emails(text)
    phone = extract_phone(text)

    # 3. Reduce text size (to save AI tokens)
    # AI cannot handle very large input efficiently
    clean_text = text[:3000]

    # 4. AI Prompt (VERY IMPORTANT)
    prompt = f"""
    Extract the following details from the text:

    - company_name
    - address
    - core_service
    - target_customer
    - probable_pain_point
    - outreach_opener

    Rules:
    - If not found, return "N/A"
    - Do NOT hallucinate (do not guess)
    - Keep answers short and clear

    TEXT:
    {clean_text}

    Return JSON only.
    """

    try:
        # Send request to AI model
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        # Convert AI response (string) into dictionary
        ai_output = json.loads(response.choices[0].message.content)

    except Exception as e:
        print(f"AI error: {e}")
        ai_output = {}

    # 5. Return final structured output (VERY STRICT FORMAT)
    return {
        "website_name": url,
        "company_name": ai_output.get("company_name", "N/A"),
        "address": ai_output.get("address", "N/A"),
        "mobile_number": phone,
        "mail": emails if emails else [],
        "core_service": ai_output.get("core_service", "N/A"),
        "target_customer": ai_output.get("target_customer", "N/A"),
        "probable_pain_point": ai_output.get("probable_pain_point", "N/A"),
        "outreach_opener": ai_output.get("outreach_opener", "N/A")
    }



In [13]:
# ========= 9. MAIN EXECUTION =========
if __name__ == "__main__":
    # 👉 Replace with provided company URLs
    urls = [
        "https://example1.com",
        "https://example2.com"
    ]

    results = []

    for url in urls:
        try:
            data = enrich_company(url)
            results.append(data)
        except Exception as e:
            print(f"Error processing {url}: {e}")

    # Save results into a JSON file
    with open("results.json", "w") as f:
        json.dump(results, f, indent=2)

    # Print results for evaluation
    print("\n=== FINAL OUTPUT ===\n")
    for r in results:
        print(r)

AI error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
AI error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

=== FINAL OUTPUT ===

{'website_name': 'https://example1.com', 'company_name': 'N/A', 'address': 'N/A', 'mobile_number': 'N/A', 'mail': [], 'core_service': 'N/A', 'target_customer': 'N/A', 'probable_pain_point': 'N/A', 'outreach_opener': 'N/A'}
{'website_name': 'https://example2.com', 'company_name': 'N/A', 'address': 'N/A', 'mobile_number': 'N/A', 'mail': [], 'core_service': 'N/A', 'target_customer': 'N/A', 'probable_pain_point': 'N/A', 'outreach_opener': 'N/A'}
